# Train τα 3 μοντέλα (UCI, MDVR, Iyer)

- **UCI model**: trained σε UCI published features (sustained vowel /a/). Για το Part 1 του test (αααα).
- **Iyer model**: trained σε δικά μας extracted features από Iyer 2023 (sustained vowel /a/). Για το Part 1 επίσης - τρίτο επίπεδο.
- **MDVR model**: trained σε δικά μας extracted features από MDVR-KCL (reading task). Για το Part 2 (ελληνικό κείμενο).

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, cross_val_score, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, confusion_matrix

from src.features import FEATURE_NAMES

MODELS = Path('../models')
MODELS.mkdir(exist_ok=True)

def make_pipe():
    return Pipeline([
        ('scaler', RobustScaler()),
        ('clf', RandomForestClassifier(
            n_estimators=300, random_state=42, n_jobs=-1,
            class_weight='balanced'
        ))
    ])

def evaluate(pipe, X, y, groups, n_splits=5):
    cv = GroupKFold(n_splits=min(n_splits, len(np.unique(groups))))
    y_proba = cross_val_predict(pipe, X, y, cv=cv, groups=groups, method='predict_proba')[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)
    print(f'  Accuracy: {accuracy_score(y, y_pred):.3f}')
    print(f'  F1:       {f1_score(y, y_pred):.3f}')
    print(f'  MCC:      {matthews_corrcoef(y, y_pred):.3f}')
    cm = confusion_matrix(y, y_pred)
    print(f'  CM: TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}')
    print(f'  Threshold scan:')
    print(f'    thr   acc    HC-recall  PD-recall')
    for t in [0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
        p = (y_proba >= t).astype(int)
        cm = confusion_matrix(y, p)
        hc_r = cm[0,0] / cm[0].sum() if cm[0].sum() else 0
        pd_r = cm[1,1] / cm[1].sum() if cm[1].sum() else 0
        print(f'    {t:.2f}  {accuracy_score(y, p):.3f}  {hc_r:.3f}      {pd_r:.3f}')

## Model 1: UCI (sustained vowel)

Lite μοντέλο που χρησιμοποιεί τα 55 features που μπορούμε να αναπαράγουμε live.

In [2]:
uci = pd.read_csv('../data/uci/pd_speech_features.csv', header=1)
X_uci = uci[FEATURE_NAMES]
y_uci = uci['class'].values
g_uci = uci['id'].values

print(f'UCI: {X_uci.shape}, classes {pd.Series(y_uci).value_counts().to_dict()}, subjects {len(np.unique(g_uci))}')
print('\n=== UCI model ===')
pipe_uci = make_pipe()
evaluate(pipe_uci, X_uci, y_uci, g_uci, n_splits=10)

pipe_uci.fit(X_uci, y_uci)
joblib.dump(pipe_uci, MODELS / 'uci_model.joblib')
print('Saved uci_model.joblib')

UCI: (756, 55), classes {1: 564, 0: 192}, subjects 252

=== UCI model ===


  Accuracy: 0.795
  F1:       0.874
  MCC:      0.380
  CM: TN=62, FP=130, FN=25, TP=539
  Threshold scan:
    thr   acc    HC-recall  PD-recall
    0.35  0.770  0.130      0.988
    0.40  0.780  0.203      0.977
    0.45  0.787  0.255      0.968
    0.50  0.795  0.323      0.956
    0.55  0.794  0.385      0.933
    0.60  0.795  0.458      0.910


Saved uci_model.joblib


## Model 2: Iyer 2023 (sustained vowel - third safety layer)

In [3]:
iyer = pd.read_csv('../data/iyer/iyer_features.csv')
X_iyer = iyer[FEATURE_NAMES]
y_iyer = iyer['class'].values
g_iyer = iyer['subject'].values  # κάθε file = ξεχωριστό subject

print(f'Iyer: {X_iyer.shape}, classes {pd.Series(y_iyer).value_counts().to_dict()}')
print('\n=== Iyer model ===')
pipe_iyer = make_pipe()
evaluate(pipe_iyer, X_iyer, y_iyer, g_iyer, n_splits=5)

pipe_iyer.fit(X_iyer, y_iyer)
joblib.dump(pipe_iyer, MODELS / 'iyer_model.joblib')
print('Saved iyer_model.joblib')

Iyer: (81, 55), classes {0: 41, 1: 40}

=== Iyer model ===


  Accuracy: 0.728
  F1:       0.725
  MCC:      0.457
  CM: TN=30, FP=11, FN=11, TP=29
  Threshold scan:
    thr   acc    HC-recall  PD-recall
    0.35  0.580  0.390      0.775
    0.40  0.593  0.439      0.750
    0.45  0.617  0.512      0.725
    0.50  0.728  0.732      0.725
    0.55  0.667  0.780      0.550
    0.60  0.667  0.829      0.500
Saved iyer_model.joblib


## Model 3: MDVR-KCL (reading task)

In [4]:
mdvr = pd.read_csv('../data/mdvr_kcl/mdvr_features.csv')
X_mdvr = mdvr[FEATURE_NAMES]
y_mdvr = mdvr['class'].values
g_mdvr = mdvr['subject'].values

print(f'MDVR: {X_mdvr.shape}, classes {pd.Series(y_mdvr).value_counts().to_dict()}')
print('\n=== MDVR model ===')
pipe_mdvr = make_pipe()
evaluate(pipe_mdvr, X_mdvr, y_mdvr, g_mdvr, n_splits=5)

pipe_mdvr.fit(X_mdvr, y_mdvr)
joblib.dump(pipe_mdvr, MODELS / 'mdvr_model.joblib')
print('Saved mdvr_model.joblib')

MDVR: (73, 55), classes {0: 42, 1: 31}

=== MDVR model ===


  Accuracy: 0.671
  F1:       0.600
  MCC:      0.322
  CM: TN=31, FP=11, FN=13, TP=18
  Threshold scan:
    thr   acc    HC-recall  PD-recall
    0.35  0.630  0.548      0.742
    0.40  0.658  0.619      0.710
    0.45  0.685  0.738      0.613
    0.50  0.671  0.738      0.581
    0.55  0.630  0.762      0.452
    0.60  0.603  0.810      0.323
Saved mdvr_model.joblib


## Σύνοψη

In [5]:
import os
for f in sorted(MODELS.glob('*.joblib')):
    print(f'{f.name:30s} {os.path.getsize(f) / 1024:.1f} KB')

iyer_model.joblib              727.1 KB
mdvr_model.joblib              640.2 KB
uci_model.joblib               3649.9 KB
